# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Lecture 13: what changes in high dimensions?

This notebook follows required Sections 10.1--10.4 only: volume
scaling, boundary shells and rejection sampling, uniform sampling from
spheres and balls, and annulus concentration. Exact Gamma-function
volume formulae are optional and random projection begins in Lecture 14.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(2026)
np.set_printoptions(precision=4, suppress=True)


## How volume moves toward the boundary

In $\mathbb R^d$, $|tE|=t^d|E|$. Hence the fraction of the unit
ball in its outer shell of width $\delta\in(0,1)$ is
$$
\frac{|B_1\setminus B_{1-\delta}|}{|B_1|}
=1-(1-\delta)^d.
$$
This concerns volume proportions; it does not say every point lies
near the boundary.


In [ ]:
dimensions = np.arange(1, 201)
for delta in (0.01, 0.03, 0.10):
    plt.plot(dimensions, 1 - (1 - delta) ** dimensions,
             label=fr"$\delta={delta}$")
plt.xlabel("dimension")
plt.ylabel("outer-shell volume fraction")
plt.ylim(0, 1.02)
plt.legend()
plt.show()
print("d=100, delta=0.03:", 1 - 0.97**100)


## Why rejection sampling becomes inefficient

Since $B_1\subseteq[-1,1]^d$, cube rejection accepts with probability
$|B_1|/2^d$. The estimate below decreases rapidly. A displayed zero
means this finite run saw no accepted points, not that the true
probability is zero.


In [ ]:
trials = 80_000
rejection_dimensions = np.arange(2, 13)
acceptance = []
for d in rejection_dimensions:
    points = rng.uniform(-1, 1, size=(trials, d))
    acceptance.append(np.mean(np.sum(points**2, axis=1) < 1))
plt.semilogy(rejection_dimensions,
             np.maximum(acceptance, 1 / trials), "o-")
plt.xlabel("dimension")
plt.ylabel("estimated acceptance probability")
plt.show()
print(dict(zip(rejection_dimensions, np.round(acceptance, 6))))


## Choosing a direction uniformly

Projecting a uniform point from a square onto the circle is not uniform
in angle: corner directions receive more mass. A spherical Gaussian is
rotationally symmetric, so $Z/|Z|$ is uniform on the sphere and is
independent of $|Z|$.


In [ ]:
n = 40_000
square = rng.uniform(-1, 1, size=(n, 2))
square /= np.linalg.norm(square, axis=1, keepdims=True)
gaussian = rng.normal(size=(n, 2))
gaussian /= np.linalg.norm(gaussian, axis=1, keepdims=True)
square_angle = np.arctan2(square[:, 1], square[:, 0])
gaussian_angle = np.arctan2(gaussian[:, 1], gaussian[:, 0])

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
axes[0].hist(square_angle, bins=36, density=True)
axes[0].set_title("square, then project")
axes[1].hist(gaussian_angle, bins=36, density=True)
axes[1].axhline(1 / (2 * np.pi), color="black", linestyle="--")
axes[1].set_title("Gaussian, then project")
for ax in axes:
    ax.set_xlabel("angle")
plt.show()


## Sampling uniformly from a ball

If $U$ is uniform on $[0,1]$ and $\Theta$ is an independent
uniform direction, then $R=U^{1/d}$ satisfies
$\mathbb P(R\leq r)=r^d$ and has density $dr^{d-1}$. Thus
$R\Theta$ is uniform in the ball. A uniform radius would put too much
mass near the centre when $d>1$.


In [ ]:
def sample_unit_sphere(n, d, rng):
    z = rng.normal(size=(n, d))
    return z / np.linalg.norm(z, axis=1, keepdims=True)

def sample_unit_ball(n, d, rng):
    direction = sample_unit_sphere(n, d, rng)
    radius = rng.uniform(size=n) ** (1 / d)
    return radius[:, None] * direction

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, d in zip(axes, (2, 10, 100)):
    sample = sample_unit_ball(25_000, d, rng)
    radii = np.linalg.norm(sample, axis=1)
    ax.hist(radii, bins=35, density=True)
    grid = np.linspace(0, 1, 300)
    ax.plot(grid, d * grid ** (d - 1), "k--", label="$d r^{d-1}$")
    ax.set_title(f"dimension {d}")
    ax.legend()
plt.tight_layout()
plt.show()
sphere = sample_unit_sphere(2_000, 25, rng)
print("maximum sphere-radius error:",
      np.max(np.abs(np.linalg.norm(sphere, axis=1) - 1)))


## Where Gaussian points concentrate

For a standard spherical Gaussian $Z\in\mathbb R^d$,
$\mathbb E|Z|^2=d$, and $|Z|/\sqrt d$ concentrates near $1$.
The notes give the exponential tail theorem. These histograms are a
diagnostic of relative concentration, not a proof.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharex=True)
for ax, d in zip(axes, (2, 10, 100)):
    z = rng.normal(size=(30_000, d))
    relative_radius = np.linalg.norm(z, axis=1) / np.sqrt(d)
    ax.hist(relative_radius, bins=50, density=True)
    ax.axvline(1, color="black", linestyle="--")
    ax.set_title(f"dimension {d}")
    print(f"d={d:3d} quantiles:",
          np.quantile(relative_radius, [0.05, 0.5, 0.95]))
plt.tight_layout()
plt.show()


## Try it yourself

1. Derive $1-(1-\delta)^d$ and find the smallest dimension where a
   shell of width $0.02$ contains at least $90\%$ of the ball.
2. Derive the density $dr^{d-1}$. Compare the correct radius with a
   uniform radius in dimensions $2$ and $20$.
3. Estimate
   $\mathbb P(\bigl||Z|/\sqrt d-1\bigr|>0.1)$ for several dimensions and plot
   the estimates. Explain why this supports but does not prove the
   annulus theorem.
